# 02 · Bronze con Auto Loader

⏱ **15 min** · Slides: 20–22 (Auto Loader, Auto Loader en código, tres formas de ingerir)

**Objetivo:** que **solo los archivos nuevos** de RAW entren a la tabla bronze, sin duplicar nunca uno.

Auto Loader corre aquí **por lotes** (`trigger(availableNow=True)`): cada ejecución procesa los archivos pendientes y se detiene.
No hay un stream encendido todo el tiempo; es una ingesta incremental que se programa como un job.

```
RAW (JSON por día)  ──► Auto Loader (cloudFiles, por lotes)  ──► bronze_schedule (Delta, con CDF)
                         lee: autoloader_read()                  escribe: autoloader_write() con MERGE (upsert)
```

| Paso | Qué haces |
|---|---|
| 1 | Crear bronze con Change Data Feed activado |
| 2 | **Lectura**: Auto Loader detecta y lee los archivos nuevos |
| 3 | **Escritura**: cada lote entra a bronze con `MERGE` (upsert) |
| 4 | Ver qué archivos ya leyó Auto Loader (`cloud_files_state`) |
| 5 | Re-ejecutar: no entra nada dos veces |
| 6 | Llega un día nuevo: entra solo ese archivo |
| 7 | Llega una corrección: actualiza, no duplica |
| 8 | `_rescued_data`: nada se pierde |

**Requiere:** notebook 01.

In [0]:
%run ./00_Setup

## Paso 0 · Empezar limpio
Para poder repetir el notebook. RAW se conserva.

In [0]:
reset_pipeline()

## Paso 1 · Crear bronze con Change Data Feed activado

La tabla se crea **antes** de la primera carga, con la propiedad `delta.enableChangeDataFeed = true`.
Así cada cambio queda registrado desde la versión 0 y el notebook 03 podrá leerlos.

In [0]:
create_bronze_table()

In [0]:
BRONZE

In [0]:
display(spark.sql(f"SHOW TBLPROPERTIES {BRONZE}"))

In [0]:


display(spark.sql(f"SHOW TBLPROPERTIES {BRONZE}").where("key = 'delta.enableChangeDataFeed'"))

## Paso 2 · LECTURA · Auto Loader detecta los archivos nuevos

`cloudFiles` es la fuente de Structured Streaming que vigila una carpeta.
Es el mismo código de `autoloader_read()` en `00_Setup`, escrito aquí para verlo completo.

In [0]:
SCHEMA_PATH

In [0]:
lectura = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")                              # cada archivo es un arreglo JSON
    .option("cloudFiles.schemaLocation", SCHEMA_PATH)         # dónde guarda el esquema
    .option("cloudFiles.rescuedDataColumn", "_rescued_data")  # lo que no encaje en el esquema va aquí
    .schema(ESQUEMA_EPISODIOS)                                # esquema explícito (regla del módulo 2)
    .load(RAW_PATH)
    .select("*",
            F.col("_metadata.file_name").alias("source_file"),   # de qué archivo viene cada fila
            F.current_timestamp().alias("ingested_at")))

print("¿Es un stream?", lectura.isStreaming)
lectura.printSchema()

👀 **Qué observar:** todavía no se leyó nada: un stream es un plan, igual que un DataFrame *lazy*.
Fíjate en las tres columnas extra: `_rescued_data`, `source_file` e `ingested_at`.

## Paso 3 · ESCRITURA · cada lote entra a bronze con `MERGE` (upsert)

- `foreachBatch(upsert_to_bronze)`: cada lote del stream pasa por una función que hace `MERGE` en bronze por `id`:
  **inserta** los episodios nuevos y **actualiza** los que ya existían. Una corrección que llegue en un archivo nuevo no duplica la fila.
- `checkpointLocation`: la **memoria** de Auto Loader (qué archivos ya procesó).
- `trigger(availableNow=True)`: procesa todo lo pendiente y **se detiene**. Es Auto Loader **por lotes**: sin stream continuo, se programa como un job.

In [0]:
def upsert_to_bronze(lote, lote_id):
    # Un lote = los archivos nuevos que Auto Loader detectó. MERGE exige una sola fila por clave.
    from delta.tables import DeltaTable
    lote = lote.dropDuplicates(["id"])
    (DeltaTable.forName(lote.sparkSession, qualified_name(BRONZE)).alias("t")
        .merge(lote.alias("s"), "t.id = s.id")
        .whenMatchedUpdateAll()           # ya existía: se actualiza
        .whenNotMatchedInsertAll()        # es nuevo: se inserta
        .execute())

query = (lectura.writeStream
    .foreachBatch(upsert_to_bronze)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(availableNow=True)
    .start())

query.awaitTermination()
print("Filas en bronze:", spark.table(BRONZE).count())

In [0]:
display(spark.table(BRONZE).groupBy("source_file").count().orderBy("source_file"))

In [0]:
BRONZE

In [0]:
%sql
DESCRIBE HISTORY bronze_schedule

In [0]:
show_history(BRONZE)

👀 **Qué observar:** una fila por episodio, agrupadas por archivo de origen. En el historial, la versión 0 es el
`CREATE TABLE` y la 1 es un `MERGE` con `numTargetRowsInserted` igual a todas las filas: así se ve una carga de Auto Loader
con upsert en el log de Delta.

## Paso 4 · ¿Qué archivos ya leyó Auto Loader?

La función `cloud_files_state('<checkpoint>')` muestra el estado de cada archivo en la memoria del stream.

In [0]:
CHECKPOINT_PATH

In [0]:
display(spark.sql(f"""
    SELECT path, size, discovery_time, commit_time
    FROM cloud_files_state('{CHECKPOINT_PATH}')
    ORDER BY path
"""))

👀 **Qué observar:** un registro por archivo de RAW. Esa lista es la que impide que un archivo entre dos veces.

## Paso 5 · Re-ejecutar: no entra nada dos veces

`run_bronze()` hace lectura + escritura con las mismas funciones del setup.

In [0]:
version_antes = table_version(BRONZE)
version_antes

In [0]:
version_antes = table_version(BRONZE)
run_bronze()
print("Versión antes:", version_antes, "| después:", table_version(BRONZE))

👀 **Qué observar:** sin archivos nuevos y la versión no cambia: no hubo ningún commit.
Compara con el módulo 2: un `append` con comodines habría duplicado todo.

## Paso 6 · Llega un día nuevo

`next_day()` descarga el día siguiente al último archivo de RAW.

In [0]:
nuevo = next_day()
download_days(nuevo, 1)
run_bronze()

In [0]:
CHECKPOINT_PATH

In [0]:
display(spark.sql(f"""
    SELECT path, size, discovery_time, commit_time
    FROM cloud_files_state('{CHECKPOINT_PATH}')
    ORDER BY commit_time DESC
"""))

In [0]:
show_history(BRONZE)

👀 **Qué observar:** entró **solo** el archivo nuevo (mira `discovery_time`), bronze creció una versión y los
archivos anteriores no se volvieron a leer.

## Paso 7 · Llega una corrección: actualiza, no duplica

La fuente reenvía un episodio con la duración corregida. Llega como **un archivo nuevo** (Auto Loader lo detecta)
y el `MERGE` lo aplica sobre la fila que ya existía.

In [0]:
episodio = spark.table(BRONZE).select("id").orderBy("id").first()["id"]
antes = spark.table(BRONZE).count()

write_correction(episodio, runtime=99, nombre="episodio")

In [0]:
episodio = spark.table(BRONZE).select("id").orderBy("id").first()["id"]
antes = spark.table(BRONZE).count()

write_correction(episodio, runtime=99, nombre="episodio")
run_bronze()
print("Filas en bronze antes:", antes, "| después:", spark.table(BRONZE).count())

In [0]:
display(spark.table(BRONZE).where(F.col("id") == episodio).select("id", "name", "runtime", "source_file", "ingested_at"))

In [0]:
show_history(BRONZE)

👀 **Qué observar:** el conteo no cambió: el episodio corregido **actualizó** su fila (`numTargetRowsUpdated = 1`) y
`source_file` ahora apunta al archivo de corrección. Con un `append` tendríamos dos filas para el mismo episodio.
Como bronze tiene CDF, esa actualización quedó registrada: la veremos en el notebook 03.

## Paso 8 · `_rescued_data`: lo que no estaba en el esquema no se pierde

El esquema explícito solo declara los campos que usamos; el resto del JSON (`url`, `image`, `_links`...) queda en `_rescued_data`.

In [0]:
display(spark.table(BRONZE)
        .select("id", "name", F.get_json_object("_rescued_data", "$.url").alias("url_rescatada"), "_rescued_data")
        .limit(3))

👀 **Qué observar:** la columna guarda un JSON con los campos que no encajaron. Si un día la API agrega un campo, no se pierde:
queda aquí hasta que decidas sumarlo al esquema.

## ✔ Checkpoint

- [ ] Sé qué hace cada opción de `cloudFiles` (format, schemaLocation, rescuedDataColumn, checkpointLocation, availableNow)
- [ ] Vi en `cloud_files_state` los archivos que Auto Loader ya leyó
- [ ] Re-ejecutar no duplicó y un día nuevo entró solo
- [ ] Una corrección que llegó como archivo nuevo actualizó la fila en lugar de duplicarla (upsert)
- [ ] Sé dónde queda lo que no está en el esquema (`_rescued_data`)

**Siguiente:** `03_Silver_con_CDF`